## MODELADO FISICO  (init.sql) COLUMNA VERTEBRAR DE MODELO DE NEGOCIO

Ya tienes la base de datos configurada ``(init.sql)``, la conexión central ``(database.py)``, los modelos ORM físicos ``(models_db.py)`` dentro de la capa de infraestructura de cada módulo y el enrutador principal ``(main.py)`` uniendo todo. Esto cubre la infraestructura inicial.  

Como el usuario proporcionó una imagen con la estructura de directorios y solicitó continuar avanzando en el contexto del proyecto en español, procedemos en dicho idioma de acuerdo con las pautas.    

Para consolidar la arquitectura de **Domain-Driven Design (DDD)** en tus directorios y que cada módulo tenga vida funcional completa, el siguiente paso clave es **implementar el flujo de datos de extremo a extremo** conectando las carpetas que faltan dentro de cada módulo **(domain, application, infrastructure y presentation)**.    

**¿Qué archivos y carpetas debes integrar ahora en cada módulo?**    
Tomando como referencia el módulo auth (y replicándolo en **order, payment, restaurant, tracking**), la estructura interna debe llenarse con los siguientes componentes esenciales:

### Paso a paso para completar los componentes :    
**1. En la capa** ``domain/repositories.py`` (El **Contrato**)
Define qué operaciones se permiten sobre los datos sin importar la base de datos:

In [ ]:
from abc import ABC, abstractmethod
from typing import Optional

class IAuthRepository(ABC):
    @abstractmethod
    def obtener_por_email(self, email: str):
        pass

    @abstractmethod
    def guardar(self, usuario):
        pass

2. **En la capa** ``infrastructure/repository_impl.py`` **(La Concreción)**
Implementa ese contrato usando la sesión de SQLAlchemy y tu ``models_db.py``:

In [ ]:
from sqlalchemy.orm import Session
from src.modules.auth.domain.repositories import IAuthRepository
from src.modules.auth.infrastructure.models_db import UsuarioModel

class AuthRepositorySqlAlchemy(IAuthRepository):
    def __init__(self, db: Session):
        self.db = db

    def obtener_por_email(self, email: str):
        return self.db.query(UsuarioModel).filter(UsuarioModel.email == email).first()

    def guardar(self, usuario: UsuarioModel):
        self.db.add(usuario)
        self.db.commit()
        self.db.refresh(usuario)
        return usuario

3. **En la capa** ``application/services.py`` **(El Servicio / Caso de Uso)**
Aplica las reglas de negocio puras y utiliza el repositorio a través de su interfaz:

In [ ]:
from sqlalchemy.orm import Session
from src.modules.auth.infrastructure.repository_impl import AuthRepositorySqlAlchemy
from src.modules.auth.domain.models import Usuario
from src.modules.auth.domain.exceptions import EmailYaRegistradoException
from src.modules.auth.presentation.schemas import UsuarioCreate

class AuthService:
    def __init__(self, db: Session):
        self.repository = AuthRepositorySqlAlchemy(db)

    def registrar_usuario(self, data: UsuarioCreate) -> Usuario:
        # 1. Verificar si el email ya existe a través del contrato
        usuario_existente = self.repository.obtener_por_email(data.email)
        if usuario_existente:
            raise EmailYaRegistradoException(f"El correo {data.email} ya está registrado en el sistema.")

        # 2. Instanciar la Entidad Pura de Dominio (Independiente de SQLAlchemy)
        nuevo_usuario = Usuario(
            nombre_usuario=data.nombre_usuario,
            email=data.email,
            contrasenia=data.contrasenia,  # Idealmente hasheada aquí o en un servicio de dominio
            rol_id=data.rol_id,
            estado_id=data.estado_id
        )

        # 3. Guardar la entidad pura mediante el repositorio
        return self.repository.guardar(nuevo_usuario)

4. **En la capa** ``presentation/schemas.py`` y ``router.py`` **(La Exposición HTTP)**
``schemas.py``: Define los modelos de Pydantic (``UsuarioCreate, UsuarioResponse``).

``router.py``: Expone los endpoints limpios inyectando la sesión de base de datos ``(Depends(get_db))``.

Tomaremos para iniciar, el **módulo auth** como plantilla de referencia oficial es la mejor estrategia pedagógica para tus estudiantes. Una vez que dominen el flujo completo de este módulo, replicarlo en los demás (``restaurant, order, payment, tracking``) será mucho más intuitivo y metódico.

A continuación, te presento el desarrollo completo y limpio de cada archivo que conforma la plantilla estándar bajo los principios de **Domain-Driven Design (DDD)** para ``FastAPI`` y ``SQLAlchemy``.

**1. Capa de Presentación:** ``src/modules/auth/presentation/schemas.py``   
(**Nota:** Este archivo se ubica dentro de la carpeta presentation o junto al ``router.py``). Define los **DTOs** (``ata Transfer Objects``) utilizando Pydantic para validar estrictamente la entrada de datos (lo que envía el cliente) y la salida (lo que responde la ``API``).


In [ ]:
from pydantic import BaseModel, EmailStr

class UsuarioCreate(BaseModel):
    nombre_usuario: str
    email: EmailStr
    contrasenia: str
    rol_id: int
    estado_id: int

class UsuarioResponse(BaseModel):
    usuario_id: int
    nombre_usuario: str
    email: str
    rol_id: int
    estado_id: int

    class Config:
        from_attributes = True # Permite mapear modelos ORM directamente a Pydantic

**2. Capa de Dominio:** ``src/modules/auth/domain/repositories.py``   
Define el contrato abstracto (``ABC``). La lógica de negocio y los servicios usarán esta interfaz sin enterarse de si la base de datos es ``PostgreSQL, MySQL o SQLite``.

In [ ]:
from abc import ABC, abstractmethod
from typing import Optional, List
from src.modules.auth.infrastructure.models_db import UsuarioModel

class IAuthRepository(ABC):
    
    @abstractmethod
    def obtener_por_id(self, usuario_id: int) -> Optional[UsuarioModel]:
        pass

    @abstractmethod
    def obtener_por_email(self, email: str) -> Optional[UsuarioModel]:
        pass

    @abstractmethod
    def guardar(self, usuario: UsuarioModel) -> UsuarioModel:
        pass

    @abstractmethod
    def listar_todos(self) -> List[UsuarioModel]:
        pass

**3. Capa de Infraestructura:** ``src/modules/auth/infrastructure/repository_impl.py``   
Implementa el contrato anterior utilizando la sesión activa de ``SQLAlchemy`` y el **modelo físico** ``(models_db.py)``.

In [ ]:
from sqlalchemy.orm import Session
from typing import Optional, List
from src.modules.auth.domain.repositories import IAuthRepository
from src.modules.auth.infrastructure.models_db import UsuarioModel

class AuthRepositorySqlAlchemy(IAuthRepository):
    def __init__(self, db: Session):
        self.db = db

    def obtener_por_id(self, usuario_id: int) -> Optional[UsuarioModel]:
        return self.db.query(UsuarioModel).filter(UsuarioModel.usuario_id == usuario_id).first()

    def obtener_por_email(self, email: str) -> Optional[UsuarioModel]:
        return self.db.query(UsuarioModel).filter(UsuarioModel.email == email).first()

    def guardar(self, usuario: UsuarioModel) -> UsuarioModel:
        self.db.add(usuario)
        self.db.commit()
        self.db.refresh(usuario)
        return usuario

    def listar_todos(self) -> List[UsuarioModel]:
        return self.db.query(UsuarioModel).all()

**4. Capa de Aplicación:** ``src/modules/auth/application/services.py``   
Orquesta la lógica del negocio. Valida reglas puras (por ejemplo, que el correo no esté registrado previamente) y coordina al repositorio.

In [ ]:
from sqlalchemy.orm import Session
from src.modules.auth.infrastructure.repository_impl import AuthRepositorySqlAlchemy
from src.modules.auth.infrastructure.models_db import UsuarioModel
from src.modules.auth.domain.exceptions import EmailYaRegistradoException
from src.modules.auth.presentation.schemas import UsuarioCreate

class AuthService:
    def __init__(self, db: Session):
        self.repository = AuthRepositorySqlAlchemy(db)

    def registrar_usuario(self, data: UsuarioCreate) -> UsuarioModel:
        # 1. Regla de negocio: Verificar si el email ya existe
        usuario_existente = self.repository.obtener_por_email(data.email)
        if usuario_existente:
            raise EmailYaRegistradoException(f"El correo {data.email} ya está registrado en el sistema.")

        # 2. En un entorno real, aquí harías el hash de la contraseña (ej. passlib / bcrypt)
        contrasenia_hasheada = data.contrasenia # (Simulado por simplicidad académica)

        # 3. Crear la instancia del modelo ORM de infraestructura
        nuevo_usuario = UsuarioModel(
            nombre_usuario=data.nombre_usuario,
            email=data.email,
            contrasenia=contrasenia_hasheada,
            rol_id=data.rol_id,
            estado_id=data.estado_id
        )

        # 4. Guardar a través del repositorio
        return self.repository.guardar(nuevo_usuario)

**5. Capa de Presentación:** ``src/modules/auth/presentation/router.py``
Expone los **endpoints HTTP** limpios mediante ``FastAPI``, inyectando la sesión de base de datos y manejando las excepciones de dominio.

In [ ]:
from fastapi import APIRouter, Depends, HTTPException, status
from sqlalchemy.orm import Session
from typing import List

from src.core.database import get_db
from src.modules.auth.presentation.schemas import UsuarioCreate, UsuarioResponse
from src.modules.auth.application.services import AuthService
from src.modules.auth.domain.exceptions import DomainException

router = APIRouter()

@router.post("/register", response_model=UsuarioResponse, status_code=status.HTTP_201_CREATED)
def registrar_usuario(data: UsuarioCreate, db: Session = Depends(get_db)):
    try:
        service = AuthService(db)
        usuario_creado = service.registrar_usuario(data)
        return usuario_creado
    except DomainException as e:
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=status.HTTP_500_INTERNAL_SERVER_ERROR, detail=str(e))

### Lo que complementa y sella la plantilla bajo DDD Separado:   
**6. Crear la Entidad Pura de Dominio:** ``src/modules/auth/domain/models.py``
Esta clase representa el concepto de negocio limpio, con sus reglas o atributos nativos de Python, sin importar si por debajo hay tablas, claves foráneas o motores ``SQL``.  

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Usuario:
    nombre_usuario: str
    email: str
    contrasenia: str
    rol_id: int
    estado_id: int
    usuario_id: Optional[int] = None

    def cambiar_password(self, nueva_contrasenia: str):
        """Ejemplo de regla de negocio pura dentro de la entidad."""
        if len(nueva_contrasenia) < 6:
            raise ValueError("La contraseña debe tener al menos 6 caracteres.")
        self.contrasenia = nueva_contrasenia

**3. Implementar el Mapeador en la Infraestructura:** ``src/modules/auth/infrastructure/repository_impl.py``    
Aquí es donde ocurre la magia de la arquitectura desacoplada. La implementación concreta con ``SQLAlchemy`` toma el modelo de base de datos ``(UsuarioModel)``, lo traduce a la entidad de dominio (``Usuario``) antes de entregarlo, y viceversa cuando se va a guardar en **PostgreSQL**.

In [ ]:
from sqlalchemy.orm import Session
from typing import Optional
from src.modules.auth.domain.repositories import IAuthRepository
from src.modules.auth.domain.models import Usuario
from src.modules.auth.infrastructure.models_db import UsuarioModel

class AuthRepositorySqlAlchemy(IAuthRepository):
    def __init__(self, db: Session):
        self.db = db

    def _to_domain(self, model: UsuarioModel) -> Optional[Usuario]:
        """Convierte un modelo ORM de BD a una Entidad Pura de Dominio."""
        if not model:
            return None
        return Usuario(
            usuario_id=model.usuario_id,
            nombre_usuario=model.nombre_usuario,
            email=model.email,
            contrasenia=model.contrasenia,
            rol_id=model.rol_id,
            estado_id=model.estado_id
        )

    def obtener_por_id(self, usuario_id: int) -> Optional[Usuario]:
        db_model = self.db.query(UsuarioModel).filter(UsuarioModel.usuario_id == usuario_id).first()
        return self._to_domain(db_model)

    def obtener_por_email(self, email: str) -> Optional[Usuario]:
        db_model = self.db.query(UsuarioModel).filter(UsuarioModel.email == email).first()
        return self._to_domain(db_model)

    def guardar(self, usuario: Usuario) -> Usuario:
        # Si ya tiene ID, actualizamos; si no, creamos el modelo de BD
        if usuario.usuario_id:
            db_model = self.db.query(UsuarioModel).filter(UsuarioModel.usuario_id == usuario.usuario_id).first()
            if db_model:
                db_model.nombre_usuario = usuario.nombre_usuario
                db_model.email = usuario.email
                db_model.contrasenia = usuario.contrasenia
                db_model.rol_id = usuario.rol_id
                db_model.estado_id = usuario.estado_id
        else:
            db_model = UsuarioModel(
                nombre_usuario=usuario.nombre_usuario,
                email=usuario.email,
                contrasenia=usuario.contrasenia,
                rol_id=usuario.rol_id,
                estado_id=usuario.estado_id
            )
            self.db.add(db_model)
        
        self.db.commit()
        self.db.refresh(db_model)
        return self._to_domain(db_model)

### ¿Por qué esta estructura escala mucho mejor?    
**Independencia tecnológica total:** Si el día de mañana decides migrar de ``PostgreSQL/SQLAlchemy`` a ``MongoDB``, ``DynamoDB`` o un almacenamiento en memoria para pruebas unitarias, **la capa de dominio (``domain/``) y los servicios (``application/``) no sufrirán ni una sola línea de cambio**. Solo reescribirías los adaptadores en la infraestructura.   

**Pruebas unitarias impecables:** Los estudiantes pueden probar la lógica de negocio pura creando instancias de la **clase** ``Usuario`` sin necesidad de levantar una base de datos real o **contenedores Docker**.

### ¿Cómo funciona el flujo de conexion en cada fichero?    

El cliente hace una **petición HTTP** ``(POST /api/v1/auth/register) con un JSON``.   

``FastAPI y Pydantic (schemas.py)`` interceptan el ``JSON`` y validan que cumpla con los tipos de datos correctos.   

El **Router** ``(router.py)`` **recibe los datos validados, extrae la sesión de la base de datos** ``(get_db)`` y se lo **entrega al Servicio** ``(services.py)``.    

El **Servicio** aplica la regla de negocio (ej. **"¿Existe este correo?"**) y, si todo es correcto, le ordena al Repositorio ``(repository_impl.py)`` interactuar con la base de datos ``PostgreSQL`` usando ``SQLAlchemy`` y el **modelo físico** ``(models_db.py)``.

La respuesta viaja de regreso en sentido inverso, transformándose en un **``JSON``** limpio de respuesta.